# Agent Tests

Each section below tests a different interaction mode with the LangGraph Design Agent.

**Prerequisites:**
- Rhino + Grasshopper running with the MCPServer component active (port 5100)
- `settings.py` configured (`LLM_PROVIDER`, `GEMINI_MODEL`, `PLAN_MODE`, endpoints)
- `.env.local` configured for secret keys (`GOOGLE_API_KEY`, `TAVILY_API_KEY`)
- Run from the `AgentApp/` directory (kernel cwd must be `AgentApp/`)


In [ ]:
## Setup — bootstrap path, build graph
import os, sys, textwrap

# Ensure AgentApp root is on sys.path regardless of where Jupyter launched from
AGENT_ROOT = os.path.abspath(os.getcwd())
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

# app/config.py loads .env.local (secrets) and settings.py (non-secret config)
from app.config import LLM_PROVIDER, LLM_ENDPOINT, GEMINI_MODEL, MCP_GH_ENDPOINT
print(f"Provider : {LLM_PROVIDER}")
print(f"Model    : {GEMINI_MODEL if LLM_PROVIDER == 'gemini' else LLM_ENDPOINT}")
print(f"GH MCP   : {MCP_GH_ENDPOINT}")


Provider : gemini
Model    : gemini-2.5-flash-lite
GH MCP   : http://localhost:5100


In [2]:
from graphs.main_graph import build_main_graph
from models.state import BoxState

graph = build_main_graph()
print("Graph built ✓")

def run(user_input: str, request_type=None, messages=None):
    """Invoke the graph and pretty-print the result. Returns (answer, result_dict)."""
    state = BoxState(
        request={"user_input": user_input},
        messages=list(messages or []),
        request_type=request_type,
    )
    result = graph.invoke(state, config={"recursion_limit": 50})

    answer = result.get("answer") or ""
    if isinstance(answer, list):
        answer = "\n".join(str(a) for a in answer)

    req_type     = result.get("request_type", "?")
    tool_results = result.get("tool_results") or {}
    plan_results = result.get("plan_results") or {}

    print(f"\n{'═'*62}")
    print(f"  type : {req_type}")
    print(f"{'─'*62}")
    if answer:
        for line in textwrap.wrap(answer, 58):
            print(f"  {line}")
    if tool_results:
        print(f"{'─'*62}")
        print("  tool_results:")
        for k, v in tool_results.items():
            print(f"    [{k}] {str(v)[:120]}")
    if plan_results:
        print(f"{'─'*62}")
        print("  plan_results:")
        for k, v in plan_results.items():
            print(f"    [{k}] {str(v)[:120]}")
    print(f"{'═'*62}")
    return answer, result

c:\Users\Admin\.conda\envs\311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  - Failed to load 'get_scene_info': Fields must not use names with leading underscores; e.g., use 'no_params' instead of '_no_params'.


  [llm] provider = gemini  model = gemini-2.5-flash-lite
Graph built ✓


---
## 2 · Chain Rhino Commands Together

Tests routing to `use_tool` → `execute_gh_tool` with the **`run_rhino_command`** native tool called back-to-back.  
Each message carries the previous conversation so the agent knows what was already created.

In [3]:
# ── Step 1: create a box via Rhino command ───────────────────────────────────
msgs = []
answer1, res1 = run(
    "Run the Rhino command to create a box at origin with width=10, depth=8, height=5",
    request_type="use_tool",
    messages=msgs,
)
msgs += [{"role": "user", "content": "create box"}, {"role": "assistant", "content": answer1}]

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...
  ┊ LLM thought: Executing `run_rhino_command(command='_-Box 0,0,0 10,8,5')`

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
  Executing `run_rhino_command(command='_-Box 0,0,0
  10,8,5')`
══════════════════════════════════════════════════════════════


In [4]:
# ── Step 2: move the most recently created object ────────────────────────────
answer2, res2 = run(
    "Now move the box 5 units in the X direction using a Rhino command",
    request_type="use_tool",
    messages=msgs,
)
msgs += [{"role": "user", "content": "move box"}, {"role": "assistant", "content": answer2}]

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════


In [5]:
# ── Step 3: capture the viewport to verify the result ────────────────────────
answer3, res3 = run(
    "Capture the Rhino viewport so I can see what was created",
    request_type="use_tool",
    messages=msgs,
)
msgs += [{"role": "user", "content": "capture viewport"}, {"role": "assistant", "content": answer3}]

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════


---
## 3 · Use a GH Tool

Tests routing to `use_tool` → `execute_gh_tool` where the agent picks a `.gh`-backed tool  
(e.g. `draw_cylinder` from the image).  
Make sure the MCPServer is active and the `.gh` file is in the watched folder.

In [6]:
# Show what GH tools are currently loaded from the MCP server
from tools import TOOL_CLASSES
if TOOL_CLASSES:
    print(f"{len(TOOL_CLASSES)} GH tool(s) loaded:")
    for t in TOOL_CLASSES:
        print(f"  • {t.name:30s} — {t.description[:70]}")
else:
    print("⚠ No GH tools loaded — is the Grasshopper MCPServer component running?")

6 GH tool(s) loaded:
  • capture_viewport               — Captures the active Rhino 3D viewport as a PNG and returns it as a bas
  • get_selected_geometry          — Returns details about every currently selected object in the Rhino sce
  • run_rhino_command              — Runs any Rhino command string exactly as if the user typed it in the c
  • list_rhino_commands            — Lists all Rhino commands registered in this session with their English
  • get_command_help               — Returns the syntax documentation for a Rhino command by reading the lo
  • tools                          — This tool create Cylinder


In [7]:
# ── Draw a cylinder using the GH draw_cylinder tool ─────────────────────────
answer_gh, res_gh = run(
    "Draw a cylinder with radius 30 and height 20 at the origin",
    request_type="use_tool",
)

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...
  ┊ LLM thought: Executing `run_rhino_command(command='_-Cylinder _Radius=30
                 _Height=20 _Center=0,0,0')`

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
  Executing `run_rhino_command(command='_-Cylinder
  _Radius=30 _Height=20 _Center=0,0,0')`
══════════════════════════════════════════════════════════════


In [8]:
# ── Reload GH tools (use after changing .gh files while notebook is open) ────
from tools import reload_mcp_tools
tools = reload_mcp_tools()
print(f"Reloaded — {len(tools)} tool(s) available")

  - Failed to load 'get_scene_info': Fields must not use names with leading underscores; e.g., use 'no_params' instead of '_no_params'.


Reloaded — 6 tool(s) available


---
## 4 · Use Native Tools

Native tools are built-in C# tools registered in `NativeToolRegistry`:
`capture_viewport`, `get_scene_info`, `get_selected_geometry`, `run_rhino_command`, `list_rhino_commands`, `get_command_help`.  
No `.gh` file needed — they are always available when the MCPServer is running.

In [9]:
# ── 4a: get_scene_info — list all objects in the Rhino document ──────────────
answer_info, _ = run(
    "What objects are currently in the Rhino scene? Use get_scene_info.",
    request_type="use_tool",
)

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...
  ┊ LLM thought: I am sorry, but there is no `get_scene_info` function available. The
                 available tools are: `capture_viewport`, `get_selected_geometry`,
                 `run_rhino_command`, `list_rhino_commands`, `get_command_help`, and
                 `tools`.

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
  I am sorry, but there is no `get_scene_info` function
  available. The available tools are: `capture_viewport`,
  `get_selected_geometry`, `run_rhino_command`,
  `list_rhino_commands`, `get_command_help`, and `tools`.
══════════════════════════════════════════════════════════════


In [10]:
# ── 4b: capture_viewport — screenshot + VLM visual reasoning ────────────────
answer_cap, _ = run(
    "Capture the Rhino viewport and describe what geometry you see",
    request_type="use_tool",
)

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════


In [11]:
# ── 4c: list_rhino_commands + get_command_help ───────────────────────────────
answer_cmds, _ = run(
    "List available Rhino commands and explain what the Box command does",
    request_type="use_tool",
)

  ┊ pre-set: use_tool
  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════


---
## 5 · Make a Plan (multi-tool chaining)

Forces `request_type="plan"` so the agent runs through `planner → plan_step (loop) → plan_summary`.  
The LLM decomposes the request, selects tools in order, passes results between steps, then writes a summary.

In [12]:
# ── 5a: plan — draw cylinder then capture to verify ─────────────────────────
answer_plan1, res_plan1 = run(
    "Draw a cylinder with radius 15 and height 30, then capture the viewport to confirm it looks right",
    request_type="plan",
)
print("\nFull plan steps:")
for k, v in (res_plan1.get("plan_results") or {}).items():
    print(f"  {k}: {str(v)[:150]}")

  ┊ pre-set: plan

────────────────────────────────────────────────────────────────────────
  ▶  PLAN MODE  —  decomposing task...
────────────────────────────────────────────────────────────────────────
  ┊ request: Draw a cylinder with radius 15 and height 30, then capture the
             viewport to confirm it looks right
  ┊ 2-step plan:
  ┊   [1] tools  —  Create a cylinder with a radius of 15 and a height of 30.
  ┊   [2] capture_viewport  —  Capture the current viewport to visually confirm the cylinder's creation and placement.

  ── Step 1/2: Create a cylinder with a radius of 15 and a height of 30.
  ┊ asking LLM for tool arguments...
  ┊ LLM thought: CALL `tools.create_cylinder(radius=15, height=30)`
  ┊ step 1: CALL `tools.create_cylinder(radius=15, height=30)`

  ── Step 2/2: Capture the current viewport to visually confirm the cylinder's creation and placement.
  ┊ asking LLM for tool arguments...
  ┊ LLM thought: The cylinder has been created. Now, I will capture the vie

In [13]:
# ── 5b: plan — multi-geometry scene build ───────────────────────────────────
# Asks the agent to create several shapes in sequence and inspect the final scene.
answer_plan2, res_plan2 = run(
    "Create a box at origin (10x8x5), then create a cylinder with radius 3 and height 10 "
    "placed at 20,0,0, then get the scene info to confirm both objects exist",
    request_type="plan",
)
print("\nFull plan steps:")
for k, v in (res_plan2.get("plan_results") or {}).items():
    print(f"  {k}: {str(v)[:150]}")

  ┊ pre-set: plan

────────────────────────────────────────────────────────────────────────
  ▶  PLAN MODE  —  decomposing task...
────────────────────────────────────────────────────────────────────────
  ┊ request: Create a box at origin (10x8x5), then create a cylinder with radius
             3 and height 10 placed at 20,0,0, then get the scene info to confirm
             both objects exist
  ┊ 3-step plan:
  ┊   [1] run_rhino_command  —  Create a box at the origin with dimensions 10x8x5.
  ┊   [2] tools  —  Create a cylinder with radius 3 and height 10 placed at 20,0,0.
  ┊   [3] get_selected_geometry  —  Get information about all objects in the scene to confirm the box and cylinder exist.

  ── Step 1/3: Create a box at the origin with dimensions 10x8x5.
  ┊ asking LLM for tool arguments...
  ┊ LLM thought: Certainly, I'll start by creating the box at the origin.  ```json {
                 "tool_code": "print(run_rhino_command('! _Box 0,0,0 10,8,5'))" } ```
  ┊ step 1: Certainl

In [14]:
# ── 5c: plan — let the classifier decide (no forced type) ───────────────────
# Drops request_type so the classifier runs first; a complex request like this
# should be classified as "plan" automatically.
answer_plan3, res_plan3 = run(
    "I need you to: 1) draw a cylinder radius=20 height=40, "
    "2) move it 15 units along X using a Rhino command, "
    "3) capture the viewport to show the final result"
)
print("\nClassified as:", res_plan3.get("request_type"))
print("\nFull plan steps:")
for k, v in (res_plan3.get("plan_results") or {}).items():
    print(f"  {k}: {str(v)[:150]}")

  ┊ LLM raw: use_tool
  ⇒ classified as: use_tool

  ┊ tools available: capture_viewport, get_selected_geometry, run_rhino_command, list_rhino_commands, get_command_help, tools
  ┊ asking LLM which tool to call...

══════════════════════════════════════════════════════════════
  type : use_tool
──────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════════

Classified as: use_tool

Full plan steps:
